In [1]:
import json
from tqdm import tqdm
from pathlib import Path

import copy
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

In [2]:
# Root Path
ROOT = Path("Amazon_products")

# Train and Test Dataset
TRAIN_CORPUS_PATH = ROOT / "train" /  "train_corpus.txt"
TEST_CORPUS_PATH = ROOT / "test" / "test_corpus.txt"

# Taxonomy
CLASSES_PATH = ROOT / "classes.txt"
HIERARCHY_PATH = ROOT / "class_hierarchy.txt"
KEYWORDS_PATH = ROOT / "class_related_keywords.txt"

In [3]:
import matplotlib.pyplot as plt
from collections import defaultdict
import itertools

# ------------------------
# Function for loads
# ------------------------

def load_lines(p: Path):
    with p.open("r", encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f]

def load_pid2text(p: Path):
    """TSV: pid \\t text  -> dict[pid]=text"""
    pid2text = {}
    with p.open("r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t", 1)
            if len(parts) == 2:
                pid, text = parts
                pid2text[pid] = text
    return pid2text

def load_classes_int(p: Path):
    class_dict = {}
    with p.open("r", encoding="utf-8") as f:
        for line in f:
            label_int, label_str = line.rstrip("\n").split("\t")
            class_dict[int(label_int)] = label_str
    return class_dict

def load_keywords(p: Path):
    keywords = {}
    with p.open("r", encoding="utf-8") as f:
        for line in f:
            key, items = line.rstrip("\n").split(":")
            item_list = [item for item in items.split(",")]
            keywords[key] = item_list
    return keywords

def load_class_graph(p: Path):
    edges = []
    with p.open("r", encoding="utf-8") as f:
        for line in f:
            p, c = map(int, line.rstrip("\n").split("\t"))
            edges.append((p, c))
    return edges

def load_json(path):
    """Load JSON file into Python object."""
    with open(path) as f:
        return json.load(f)

# ------------------------
# Visualization
# ------------------------

def plot_results(results_dict, split="valid", metric="Loss"):
    """
    Plot metric (e.g., loss) values over epochs for multiple models.

    Args:
        results_dict: dict of dicts
            Example:
                results_dict["valid"]["mlp_partial"] = [0.69, 0.65, ...]
        split: "train" | "valid" | "test"
        metric: name of the metric to display (default: Loss)
    """
    assert split in results_dict, f"{split} not in results_dict"

    plt.figure(figsize=(8, 5))

    for label, value_list in results_dict[split].items():
        plt.plot(
            range(1, len(value_list) + 1),
            value_list,
            marker="o",
            label=label
        )

    plt.title(f"{split.capitalize()} {metric} over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel(metric)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()



In [4]:
# ---------- Read-only loads ----------

train_pid2text    = load_pid2text(TRAIN_CORPUS_PATH)
test_pid2text     = load_pid2text(TEST_CORPUS_PATH)
pid2class         = load_classes_int(CLASSES_PATH)
rel_keywords      = load_keywords(KEYWORDS_PATH)
class_graph_edges = load_class_graph(HIERARCHY_PATH)

NUM_CLASSES = len(pid2class)
print(f"#train={len(train_pid2text):,}  #test={len(test_pid2text):,}")

#train=29,487  #test=19,658


# Core Class Mining

In [5]:
EMB_PATH = ROOT / "Embeddings"

train_emb_dict = load_json(EMB_PATH / "train_embeddings.json")
test_emb_dict = load_json(EMB_PATH / "test_embeddings.json")
class_emb_dict = load_json(EMB_PATH / "class_embeddings.json")
class_llm_emb_dict = load_json(EMB_PATH / "class_llm_embeddings.json")

train_doc_emb = np.stack(list(train_emb_dict.values()))   # (N_train, D)
test_doc_emb  = np.stack(list(test_emb_dict.values()))    # (N_test, D)
class_name_emb = np.stack(list(class_emb_dict.values()))  # (NUM_CLASSES, D)
class_llm_emb = np.stack(list(class_llm_emb_dict.values()))  # (NUM_CLASSES, D)

In [6]:
from collections import defaultdict

class Taxonomy:
    def __init__(self, num_classes, edges):
        self.num_classes = num_classes
        self.children = defaultdict(list)
        self.parent = {cid: None for cid in range(num_classes)}

        for p, c in edges:
            self.children[p].append(c)
            self.parent[c] = p

        self.roots = [cid for cid, p in self.parent.items() if p is None]

    def get_ancestors(self, cid):
        """cid의 조상 리스트 (자기 자신 제외, root에서부터 오름차순)."""
        res = []
        cur = self.parent[cid]
        while cur is not None:
            res.append(cur)
            cur = self.parent[cur]
        return res[::-1]

    def get_path(self, cid):
        """root → ... → cid까지 path (자기 자신 포함)."""
        return self.get_ancestors(cid) + [cid]

    def get_descendants(self, cid):
        """cid 포함 모든 자손 (자기 자신 제외할지 포함할지 선택 가능)."""
        res = []
        stack = [cid]
        while stack:
            cur = stack.pop()
            for ch in self.children[cur]:
                res.append(ch)
                stack.append(ch)
        return res


# Taxonomy 인스턴스 생성
taxonomy = Taxonomy(NUM_CLASSES, class_graph_edges)
print("Roots:", taxonomy.roots)

Roots: [0, 3, 10, 23, 40, 169]


In [7]:
sim_matrix = train_doc_emb @ class_name_emb.T   # (N_train, NUM_CLASSES)
print("sim_matrix:", sim_matrix.shape)

paths = {cid: taxonomy.get_path(cid) for cid in range(NUM_CLASSES)}
path_scores = np.zeros_like(sim_matrix, dtype=np.float32)  # (N_train, NUM_CLASSES)

for cid, path in paths.items():
    vals = sim_matrix[:, path]
    path_scores[:, cid] = vals.mean(axis=1)

print("path_scores:", path_scores.shape)

TOP_K_CORE = 3
THRESHOLD  = 0.0

core_classes_per_doc = []
for i in range(path_scores.shape[0]):
    row = path_scores[i]

    idx_sorted = np.argsort(-row)
    idx_sorted = [cid for cid in idx_sorted if row[cid] >= THRESHOLD]

    core = idx_sorted[:TOP_K_CORE]
    core_classes_per_doc.append(core)

sim_matrix: (29487, 531)
path_scores: (29487, 531)


In [8]:
def build_silver_labels(core_classes_per_doc, taxonomy, num_classes):
    """
    TaxoClass (식 7) 아이디어를 따라:
    - C_pos(i) = core classes ∪ ancestors(core)
    - negative는 BCEWithLogitsLoss에서 0으로 처리 (child-exclude는 단순화)
    """
    N = len(core_classes_per_doc)
    Y = np.zeros((N, num_classes), dtype=np.float32)

    for i, core_list in enumerate(core_classes_per_doc):
        pos = set()
        for cid in core_list:
            pos.add(cid)
            for anc in taxonomy.get_ancestors(cid):
                pos.add(anc)
        for cid in pos:
            Y[i, cid] = 1.0

    return Y

Y_silver = build_silver_labels(core_classes_per_doc, taxonomy, NUM_CLASSES)

print("Y_silver shape:", Y_silver.shape)
print("문서 0의 양성 라벨 수:", int(Y_silver[0].sum()))

Y_silver shape: (29487, 531)
문서 0의 양성 라벨 수: 4


In [11]:
df = pd.DataFrame(
    Y_silver,
    columns=[f"class {i}" for i in range(Y_silver.shape[1])]
)

SILVER_PATH = Path("Silver Core")

df.to_csv(SILVER_PATH / "Y_silver.csv", index=False)